# Linear Regression Health Costs Calculator

Completed freeCodeCamp Machine Learning with Python project. The notebook predicts medical expenses with a TensorFlow/Keras regression model and preserves the official final test cell.

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget -q https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Prepare data and train the regression model.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Convert categorical variables to numeric values.
dataset = dataset.copy()
dataset['sex'] = dataset['sex'].map({'female': 0.0, 'male': 1.0})
dataset['smoker'] = dataset['smoker'].map({'no': 0.0, 'yes': 1.0})
dataset = pd.get_dummies(dataset, columns=['region'], dtype=float)

# A few deterministic interaction terms make the nonlinear cost structure
# easier for the neural network to learn, especially the smoker effect.
dataset['age_squared'] = dataset['age'] ** 2
dataset['bmi_squared'] = dataset['bmi'] ** 2
dataset['smoker_age'] = dataset['smoker'] * dataset['age']
dataset['smoker_bmi'] = dataset['smoker'] * dataset['bmi']

# Required 80/20 split.
train_dataset = dataset.sample(frac=0.8, random_state=SEED)
test_dataset = dataset.drop(train_dataset.index)

# Required labels.
train_labels = train_dataset.pop('expenses').astype('float32')
test_labels = test_dataset.pop('expenses').astype('float32')

# Normalize input features using training data only.
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(np.asarray(train_dataset).astype('float32'))

output_bias = keras.initializers.Constant(float(train_labels.mean()))

model = keras.Sequential([
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, bias_initializer=output_bias)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.002),
    loss='mae',
    metrics=['mae', 'mse']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_mae',
    patience=50,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    train_labels,
    validation_split=0.2,
    epochs=600,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop]
)

print(f"Training epochs: {len(history.history['loss'])}")
print(f"Best validation MAE: {min(history.history['val_mae']):.2f}")

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
